# Исследование датасета Food.com

EDA для проекта **Meal Generator**.

Цель ноутбука — не полностью очистить исходный Food.com, а зафиксировать решения,
необходимые для формирования curated recipe corpus для первой версии приложения.

Основные вопросы:
- насколько надёжны КБЖУ и количество порций;
- как получить кандидатов для `breakfast` и `main_meal`;
- какие поля обязательны для Meal Generator v1;
- можно ли использовать `RecipeIngredientQuantities`;
- насколько сложна нормализация названий ингредиентов.

## 1. Загрузка данных

In [1]:
from pathlib import Path
import ast
import csv

import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
project_dir = Path.cwd()
candidate_data_dirs = [
    project_dir / "data",
    project_dir.parent / "data",
]

DATA_DIR = next(
    (path for path in candidate_data_dirs if path.exists()),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Каталог data не найден. Запускайте notebook из корня проекта "
        "или из каталога notebooks."
    )

parquet_files = sorted(DATA_DIR.glob("*.parquet"))
print(f"Каталог данных: {DATA_DIR.resolve()}")
print("Parquet-файлы:")
display([path.name for path in parquet_files])

datasets = {
    path.stem: pd.read_parquet(path)
    for path in parquet_files
}

recipes = datasets["recipes"]
reviews = datasets["reviews"]

for name, dataset in datasets.items():
    print(f"{name}: {dataset.shape[0]:,} строк, {dataset.shape[1]} столбцов")

Каталог данных: C:\Users\paink\Desktop\py_projects\meal_generator\data
Parquet-файлы:


['recipes.parquet', 'reviews.parquet']

recipes: 522,517 строк, 28 столбцов
reviews: 1,401,982 строк, 8 столбцов


## 2. Структура датасетов

Для дальнейшей работы используются Parquet-файлы. CSV-дубликаты не нужны в рабочем
pipeline: Parquet компактнее, сохраняет типы и удобнее для больших таблиц.

In [3]:
for name, dataset in datasets.items():
    print(f"--- {name} ---")
    display(dataset.dtypes.astype(str).rename("тип данных").to_frame())

print("Пример recipes:")
display(
    recipes[
        [
            "RecipeId",
            "Name",
            "RecipeCategory",
            "Calories",
            "FatContent",
            "CarbohydrateContent",
            "ProteinContent",
            "RecipeServings",
        ]
    ].head(5)
)

print("Пример reviews:")
display(reviews.head(5))

--- recipes ---


,тип данных
RecipeId,float64
Name,str
AuthorId,int32
AuthorName,str
CookTime,str
PrepTime,str
TotalTime,str
DatePublished,"datetime64[us, UTC]"
Description,str
Images,object


--- reviews ---


,тип данных
ReviewId,int32
RecipeId,int32
AuthorId,int32
AuthorName,str
Rating,int32
Review,str
DateSubmitted,"datetime64[us, UTC]"
DateModified,"datetime64[us, UTC]"


Пример recipes:


,RecipeId,Name,RecipeCategory,Calories,FatContent,CarbohydrateContent,ProteinContent,RecipeServings
0,38.0,Low-Fat Berry Blue Frozen Dessert,Frozen Desserts,170.9,2.5,37.1,3.2,4.0
1,39.0,Biryani,Chicken Breast,1110.7,58.8,84.4,63.4,6.0
2,40.0,Best Lemonade,Beverages,311.1,0.2,81.5,0.3,4.0
3,41.0,Carina's Tofu-Vegetable Kebabs,Soy/Tofu,536.1,24.0,64.2,29.3,2.0
4,42.0,Cabbage Soup,Vegetable,103.6,0.4,25.1,4.3,4.0


Пример reviews:


,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25 21:44:00+00:00,2000-01-25 21:44:00+00:00
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the differ...",2001-10-17 16:49:59+00:00,2001-10-17 16:49:59+00:00
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could ...,2000-02-25 09:00:00+00:00,2000-02-25 09:00:00+00:00
3,13,7435,1773,Malarkey Test,5,easily the best i have ever had. juicy flavor...,2000-03-13 21:15:00+00:00,2000-03-13 21:15:00+00:00
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28 12:51:00+00:00,2000-03-28 12:51:00+00:00


## 3. КБЖУ, servings и хвосты распределений

Для Meal Generator v1 основной nutrition-контракт — **КБЖУ на одну порцию**:
`Calories`, `ProteinContent`, `FatContent`, `CarbohydrateContent`.

Дополнительные нутриенты из Food.com сохраняются в raw dataset, но не входят
в основной domain contract v1.

In [4]:
statistics_names = {
    "count": "Количество",
    "mean": "Среднее",
    "std": "Стандартное отклонение",
    "min": "Минимум",
    "25%": "25-й перцентиль",
    "50%": "50-й перцентиль (медиана)",
    "75%": "75-й перцентиль",
    "max": "Максимум",
}

recipe_numeric_columns = [
    "AggregatedRating",
    "ReviewCount",
    "Calories",
    "FatContent",
    "CarbohydrateContent",
    "ProteinContent",
    "RecipeServings",
]

display(
    recipes[recipe_numeric_columns]
    .describe()
    .T
    .rename(columns=statistics_names)
)

print("Распределение Rating в reviews:")
display(
    reviews["Rating"]
    .describe()
    .rename(index=statistics_names)
    .to_frame("Rating")
)

,Количество,Среднее,Стандартное отклонение,Минимум,25-й перцентиль,50-й перцентиль (медиана),75-й перцентиль,Максимум
AggregatedRating,269294.0,4.632014,0.641934,1.0,4.5,5.0,5.0,5.0
ReviewCount,275028.0,5.227784,20.381347,1.0,1.0,2.0,4.0,3063.0
Calories,522517.0,484.438580,1397.116649,0.0,174.2,317.1,529.1,612854.6
FatContent,522517.0,24.614922,111.485798,0.0,5.6,13.8,27.4,64368.1
CarbohydrateContent,522517.0,49.089092,180.822062,0.0,12.8,28.2,51.1,108294.6
ProteinContent,522517.0,17.469510,40.128837,0.0,3.5,9.1,25.0,18396.2
RecipeServings,339606.0,8.606191,114.319809,1.0,4.0,6.0,8.0,32767.0


Распределение Rating в reviews:


,Rating
Количество,1.401982e+06
Среднее,4.407951e+00
Стандартное отклонение,1.272012e+00
Минимум,0.000000e+00
25-й перцентиль,4.000000e+00
50-й перцентиль (медиана),5.000000e+00
75-й перцентиль,5.000000e+00
Максимум,5.000000e+00


In [5]:
nutrition_columns = [
    "Calories",
    "ProteinContent",
    "FatContent",
    "CarbohydrateContent",
    "RecipeServings",
]
percentiles = [0.90, 0.95, 0.99, 0.995, 0.999]
rows_to_show = ["90%", "95%", "99%", "99.5%", "99.9%", "max"]

display(
    recipes[nutrition_columns]
    .describe(percentiles=percentiles)
    .loc[rows_to_show]
    .T
)

quality_checks = pd.Series(
    {
        "Calories <= 0": recipes["Calories"].le(0).sum(),
        "ProteinContent == 0": recipes["ProteinContent"].eq(0).sum(),
        "RecipeServings is null": recipes["RecipeServings"].isna().sum(),
        "RecipeServings <= 0": recipes["RecipeServings"].le(0).sum(),
        "Отрицательные БЖУ": (
            recipes[
                ["ProteinContent", "FatContent", "CarbohydrateContent"]
            ].lt(0).any(axis=1).sum()
        ),
    },
    name="Количество строк",
).astype(int)

display(quality_checks.to_frame())

,90%,95%,99%,99.5%,99.9%,max
Calories,862.8,1305.8,3642.672,5001.530,8554.8680,612854.6
ProteinContent,41.3,54.1,95.600,123.800,245.3968,18396.2
FatContent,49.4,75.6,208.100,284.684,513.7000,64368.1
CarbohydrateContent,86.8,136.8,476.600,648.842,1098.0100,108294.6
RecipeServings,16.0,24.0,48.000,60.000,108.0000,32767.0


,Количество строк
Calories <= 0,3111
ProteinContent == 0,8403
RecipeServings is null,182911
RecipeServings <= 0,0
Отрицательные БЖУ,0


In [6]:
anomaly_columns = [
    "Name",
    "RecipeCategory",
    "Calories",
    "ProteinContent",
    "FatContent",
    "CarbohydrateContent",
    "RecipeServings",
    "RecipeYield",
]

conditions = {
    "Calories > 5000": recipes["Calories"].gt(5000),
    "Calories == 0": recipes["Calories"].eq(0),
    "ProteinContent == 0": recipes["ProteinContent"].eq(0),
    "RecipeServings > 100": recipes["RecipeServings"].gt(100),
}

for condition_name, condition in conditions.items():
    print(f"{condition_name}: {int(condition.sum()):,} строк")
    display(recipes.loc[condition, anomaly_columns].head(5))

Calories > 5000: 2,615 строк


,Name,RecipeCategory,Calories,ProteinContent,FatContent,CarbohydrateContent,RecipeServings,RecipeYield
177,Chocolate Pudding,Dessert,6860.2,118.9,60.2,1579.0,NaN,1 jar
248,Curry Curlies,Lunch/Snacks,6831.4,73.2,529.1,465.0,NaN,1 batch
328,Linda'a Guadalajara - Style Burritos,Lunch/Snacks,5890.2,273.4,215.8,698.1,1.0,NaN
341,M&M and Chocolate Chews,Dessert,6541.0,61.1,339.8,858.5,1.0,NaN
372,English Sherry Trifle,Dessert,5527.5,85.6,238.5,744.8,1.0,NaN


Calories == 0: 3,111 строк


,Name,RecipeCategory,Calories,ProteinContent,FatContent,CarbohydrateContent,RecipeServings,RecipeYield
342,Cafe Latte,Beverages,0.0,0.0,0.0,0.0,1.0,NaN
433,Habanero Honey,Very Low Carbs,0.0,0.0,0.0,0.0,NaN,1 cup
434,Glowing Jell-O,Gelatin,0.0,0.0,0.0,0.0,1.0,NaN
458,Gray Girls' Easy Chicken Fingers,Chicken Breast,0.0,0.0,0.0,0.0,14.0,NaN
466,Grilled Bell Peppers,Peppers,0.0,0.0,0.0,0.0,NaN,1 batch


ProteinContent == 0: 8,403 строк


,Name,RecipeCategory,Calories,ProteinContent,FatContent,CarbohydrateContent,RecipeServings,RecipeYield
245,Rock Candy,Candy,1548.0,0.0,0.0,399.9,NaN,3 jars
247,Rock Candy II,Candy,1548.0,0.0,0.0,399.9,1.0,NaN
253,How to Make Yeast,Yeast Breads,387.0,0.0,0.0,100.0,1.0,NaN
342,Cafe Latte,Beverages,0.0,0.0,0.0,0.0,1.0,NaN
374,Spice Scones,Scones,23.6,0.0,1.3,2.9,60.0,NaN


RecipeServings > 100: 362 строк


,Name,RecipeCategory,Calories,ProteinContent,FatContent,CarbohydrateContent,RecipeServings,RecipeYield
1037,Mountain Maple Poppy Dressing,Salad Dressings,24.4,0.1,1.7,2.3,192.0,1 1/2 gallon
1318,Maryland Style Bull Roast,Meat,0.0,0.0,0.0,0.0,480.0,NaN
3341,Cream Cheese Spritz,Cheese,32.4,0.3,2.4,2.5,108.0,NaN
3388,Little-Bitty Coffee-Hazelnut Biscotti,Dessert,8.0,0.2,0.3,1.2,180.0,NaN
5316,Kathy Hirdler's Fire Camp Chili Con Carne With...,Pork,580.2,35.2,3.2,107.7,120.0,60 gallons


### Вывод

Экстремальные значения не всегда являются ошибками: среди них есть bulk-рецепты
и рецепты, где одна "порция" фактически соответствует целому batch.

Поэтому статистические выбросы не удаляются механически по перцентилям.
Для приложения используется отдельный слой **product eligibility** поверх
технической валидации.

## 4. Rating и ReviewCount

Отзывы используются как **soft ranking signal**, а не как жёсткий фильтр.
Новый или малооценённый рецепт не должен автоматически исключаться из выдачи.

In [7]:
rating_summary = pd.Series(
    {
        "Рецептов с AggregatedRating": recipes["AggregatedRating"].notna().sum(),
        "Рецептов без AggregatedRating": recipes["AggregatedRating"].isna().sum(),
        "Рецептов с ReviewCount": recipes["ReviewCount"].notna().sum(),
        "Максимальный ReviewCount": recipes["ReviewCount"].max(),
    },
    name="Количество",
)

display(rating_summary.to_frame())

print("Распределение пользовательских оценок:")
display(
    reviews["Rating"]
    .value_counts()
    .sort_index()
    .rename("Количество отзывов")
    .to_frame()
)

,Количество
Рецептов с AggregatedRating,269294.0
Рецептов без AggregatedRating,253223.0
Рецептов с ReviewCount,275028.0
Максимальный ReviewCount,3063.0


Распределение пользовательских оценок:


,Количество отзывов
Rating,
0,76248
1,16559
2,17597
3,50279
4,229217
5,1012082


## 5. Meal-type taxonomy

Food.com смешивает в `RecipeCategory` и `Keywords` несколько типов таксономии:
приём пищи, основной ингредиент, кухню, время приготовления и nutrition labels.

Поэтому `meal_type` является **производным признаком Meal Generator**, а не
готовой колонкой источника.

In [8]:
def to_list(value):
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return list(value)

    if value is None or (
        not isinstance(value, str)
        and bool(pd.isna(value))
    ):
        return []

    if not isinstance(value, str):
        return [value]

    text = value.strip()
    if not text:
        return []

    if text.startswith("c(") and text.endswith(")"):
        return next(
            csv.reader(
                [text[2:-1]],
                skipinitialspace=True,
            )
        )

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return [text]

    return (
        list(parsed)
        if isinstance(parsed, (list, tuple, set))
        else [parsed]
    )


normalized_categories = (
    recipes["RecipeCategory"]
    .fillna("")
    .str.strip()
    .str.casefold()
)

normalized_keywords = recipes["Keywords"].apply(
    lambda value: {
        str(keyword).strip().casefold()
        for keyword in to_list(value)
        if str(keyword).strip()
    }
)

keyword_values = (
    recipes["Keywords"]
    .apply(to_list)
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)
keyword_values = keyword_values[keyword_values.ne("")]

print(f"Уникальных RecipeCategory: {recipes['RecipeCategory'].nunique(dropna=True)}")
print(f"Уникальных Keywords: {keyword_values.nunique()}")

print("Топ-30 RecipeCategory:")
display(
    recipes["RecipeCategory"]
    .value_counts()
    .head(30)
    .rename("Количество рецептов")
    .to_frame()
)

print("Топ-30 Keywords:")
display(
    keyword_values
    .value_counts()
    .head(30)
    .rename("Количество рецептов")
    .to_frame()
)

Уникальных RecipeCategory: 311
Уникальных Keywords: 314
Топ-30 RecipeCategory:


,Количество рецептов
RecipeCategory,
Dessert,62072
Lunch/Snacks,32586
One Dish Meal,31345
Vegetable,27231
Breakfast,21101
Beverages,16076
Chicken,13249
Meat,13131
Breads,12804


Топ-30 Keywords:


,Количество рецептов
Keywords,
Easy,276838
< 60 Mins,149589
< 30 Mins,112229
< 4 Hours,111071
Meat,103191
< 15 Mins,89340
Healthy,83305
Vegetable,81264
Low Cholesterol,74121


In [9]:
search_pattern = r"breakfast|brunch|lunch|snack|dinner"

matched_keyword_counts = (
    keyword_values[
        keyword_values.str.contains(
            search_pattern,
            case=False,
            na=False,
        )
    ]
    .value_counts()
    .rename("Количество рецептов")
    .to_frame()
)

matched_category_counts = (
    recipes.loc[
        recipes["RecipeCategory"].str.contains(
            search_pattern,
            case=False,
            na=False,
        ),
        "RecipeCategory",
    ]
    .value_counts()
    .rename("Количество рецептов")
    .to_frame()
)

print("Meal-related Keywords:")
display(matched_keyword_counts)

print("Meal-related RecipeCategory:")
display(matched_category_counts)

Meal-related Keywords:


,Количество рецептов
Keywords,
Brunch,36278
Lunch/Snacks,12561
Breakfast,7595
Eggs Breakfast,3
Breakfast Potatoes,1
Breakfast Eggs,1


Meal-related RecipeCategory:


,Количество рецептов
RecipeCategory,
Lunch/Snacks,32586
Breakfast,21101
Brunch,598
Breakfast Eggs,3
Snacks Sweet,1
Breakfast Casseroles,1


In [10]:
meal_groups = ["Breakfast", "Brunch", "Lunch/Snacks"]

group_masks = {}
group_rows = []

for group in meal_groups:
    normalized_group = group.casefold()
    category_mask = normalized_categories.eq(normalized_group)
    keyword_mask = normalized_keywords.apply(
        lambda values: normalized_group in values
    )
    group_masks[group] = category_mask | keyword_mask

    group_rows.append(
        {
            "Группа": group,
            "RecipeCategory": int(category_mask.sum()),
            "Keywords": int(keyword_mask.sum()),
            "Пересечение": int((category_mask & keyword_mask).sum()),
            "Объединение": int(group_masks[group].sum()),
        }
    )

display(pd.DataFrame(group_rows).set_index("Группа"))

breakfast = group_masks["Breakfast"]
brunch = group_masks["Brunch"]
lunch_snacks = group_masks["Lunch/Snacks"]
at_least_one = breakfast | brunch | lunch_snacks

intersection_counts = pd.Series(
    {
        "Breakfast ∩ Brunch": (breakfast & brunch).sum(),
        "Breakfast ∩ Lunch/Snacks": (breakfast & lunch_snacks).sum(),
        "Brunch ∩ Lunch/Snacks": (brunch & lunch_snacks).sum(),
        "Breakfast ∩ Brunch ∩ Lunch/Snacks": (
            breakfast & brunch & lunch_snacks
        ).sum(),
        "Хотя бы одна из трёх групп": at_least_one.sum(),
        "Ни одной из трёх групп": (~at_least_one).sum(),
    },
    name="Количество рецептов",
).astype(int)

display(intersection_counts.to_frame())

,RecipeCategory,Keywords,Пересечение,Объединение
Группа,,,,
Breakfast,21101,7595,0,28696
Brunch,598,36278,0,36876
Lunch/Snacks,32586,12561,0,45147


,Количество рецептов
Breakfast ∩ Brunch,10329
Breakfast ∩ Lunch/Snacks,3255
Brunch ∩ Lunch/Snacks,6479
Breakfast ∩ Brunch ∩ Lunch/Snacks,1157
Хотя бы одна из трёх групп,91813
Ни одной из трёх групп,430704


### Baseline для `main_meal`

В Food.com нет явной категории `Dinner`, поэтому для v1 используется
детерминированный weak-labeling baseline.

Положительные сигналы:
`One Dish Meal`, `Chicken`, `Chicken Breast`, `Meat`, `Pork`, `Stew`.

Рецепты с явными `Breakfast` / `Brunch` сигналами исключаются из `main_meal`.

In [11]:
main_meal_signals = {
    "One Dish Meal",
    "Chicken",
    "Chicken Breast",
    "Meat",
    "Pork",
    "Stew",
}
normalized_main_meal_signals = {
    value.casefold()
    for value in main_meal_signals
}

main_category_mask = normalized_categories.isin(
    normalized_main_meal_signals
)
main_keyword_mask = normalized_keywords.apply(
    lambda values: bool(
        values & normalized_main_meal_signals
    )
)
main_meal_raw_mask = main_category_mask | main_keyword_mask

breakfast_brunch_signals = {
    "breakfast",
    "brunch",
}
breakfast_brunch_mask = (
    normalized_categories.isin(breakfast_brunch_signals)
    | normalized_keywords.apply(
        lambda values: bool(
            values & breakfast_brunch_signals
        )
    )
)

refined_main_meal_mask = (
    main_meal_raw_mask
    & ~breakfast_brunch_mask
)

main_meal_counts = pd.Series(
    {
        "Исходные main_meal candidates": main_meal_raw_mask.sum(),
        "Исключено Breakfast/Brunch": (
            main_meal_raw_mask & breakfast_brunch_mask
        ).sum(),
        "Осталось main_meal candidates": refined_main_meal_mask.sum(),
    },
    name="Количество рецептов",
).astype(int)

display(main_meal_counts.to_frame())

sample_columns = [
    "Name",
    "RecipeCategory",
    "Keywords",
    "Calories",
    "RecipeServings",
]

display(
    recipes.loc[
        refined_main_meal_mask,
        sample_columns,
    ].sample(
        n=20,
        random_state=42,
    )
)

,Количество рецептов
Исходные main_meal candidates,133280
Исключено Breakfast/Brunch,7576
Осталось main_meal candidates,125704


,Name,RecipeCategory,Keywords,Calories,RecipeServings
293861,Kickn' Shepherd's Pie,One Dish Meal,"[Potato, Vegetable, Meat, Potluck, Spicy, < 60...",602.1,NaN
347247,Lazy Cook's Golumpki (Stuffed Cabbage),One Dish Meal,"[Rice, Vegetable, Meat, Polish, European, St. ...",364.1,6.0
205908,Spinach and Meat Whole Wheat Lasagna,One Dish Meal,"[Spinach, Greens, Vegetable, Meat, European, <...",591.1,8.0
176760,Chicken and Sun-Dried Tomato Pasta,One Dish Meal,"[Chicken Breast, Chicken, Spaghetti, Poultry, ...",875.9,NaN
50223,Mexican Dinner,One Dish Meal,"[Meat, Southwestern U.S., Mexican, Very Low Ca...",405.3,NaN
449419,Noble House Citrus Beef,One Dish Meal,"[Steak, Meat, Chinese, Asian, Healthy, Savory,...",472.1,4.0
266645,Simple Chili With Polenta,One Dish Meal,"[Grains, Beans, Meat, Egg Free, Free Of..., < ...",553.1,NaN
304243,Easy Tex Mex Wontons,Pork,"[Meat, Tex Mex, Southwestern U.S., Very Low Ca...",182.5,NaN
161655,Spicy Fried Chicken,Chicken,"[Poultry, Meat, Healthy, Beginner Cook, < 4 Ho...",206.5,6.0
182002,Bower's Hunter's Chili,Deer,"[Wild Game, Beans, Meat, Low Cholesterol, Heal...",372.9,24.0


## 6. Technical eligibility для Meal Generator v1

Для v1 рецепт считается технически пригодным, если:
- есть `RecipeId` и название;
- есть список ингредиентов;
- есть инструкции;
- известно положительное количество порций;
- `Calories > 0`;
- КБЖУ заполнено и не содержит отрицательных значений.

`RecipeIngredientQuantities` **не входит** в hard requirements: ниже показано,
что количества не имеют надёжного позиционного соответствия ингредиентам.

In [12]:
def is_null_or_empty(value):
    return len(to_list(value)) == 0


recipe_id_invalid = recipes["RecipeId"].isna()
name_invalid = (
    recipes["Name"].isna()
    | recipes["Name"].fillna("").str.strip().eq("")
)
ingredient_parts_invalid = (
    recipes["RecipeIngredientParts"]
    .apply(is_null_or_empty)
)
ingredient_quantities_invalid = (
    recipes["RecipeIngredientQuantities"]
    .apply(is_null_or_empty)
)
instructions_invalid = (
    recipes["RecipeInstructions"]
    .apply(is_null_or_empty)
)
servings_invalid = (
    recipes["RecipeServings"].isna()
    | recipes["RecipeServings"].le(0)
)

macro_columns = [
    "ProteinContent",
    "FatContent",
    "CarbohydrateContent",
]
nutrition_columns = [
    "Calories",
    *macro_columns,
]

nutrition_missing = (
    recipes[nutrition_columns]
    .isna()
    .any(axis=1)
)
calories_invalid = (
    recipes["Calories"].isna()
    | recipes["Calories"].le(0)
)
macros_negative = (
    recipes[macro_columns]
    .lt(0)
    .any(axis=1)
)

technically_valid_mask = ~(
    recipe_id_invalid
    | name_invalid
    | ingredient_parts_invalid
    | instructions_invalid
    | servings_invalid
    | calories_invalid
    | nutrition_missing
    | macros_negative
)

technical_quality_counts = pd.Series(
    {
        "RecipeId invalid": recipe_id_invalid.sum(),
        "Name null / empty": name_invalid.sum(),
        "RecipeIngredientParts null / empty": ingredient_parts_invalid.sum(),
        "RecipeIngredientQuantities null / empty (не фильтр)": (
            ingredient_quantities_invalid.sum()
        ),
        "RecipeInstructions null / empty": instructions_invalid.sum(),
        "RecipeServings null / <= 0": servings_invalid.sum(),
        "Calories <= 0 / null": calories_invalid.sum(),
        "КБЖУ содержит null": nutrition_missing.sum(),
        "БЖУ содержит отрицательные значения": macros_negative.sum(),
        "Технически пригодных рецептов": technically_valid_mask.sum(),
    },
    name="Количество рецептов",
).astype(int)

display(technical_quality_counts.to_frame())

,Количество рецептов
RecipeId invalid,0
Name null / empty,0
RecipeIngredientParts null / empty,2219
RecipeIngredientQuantities null / empty (не фильтр),20
RecipeInstructions null / empty,2
RecipeServings null / <= 0,182911
Calories <= 0 / null,3111
КБЖУ содержит null,0
БЖУ содержит отрицательные значения,0
Технически пригодных рецептов,336510


In [13]:
breakfast_candidate_mask = (
    group_masks["Breakfast"]
    | group_masks["Brunch"]
)

technically_valid_breakfast_mask = (
    technically_valid_mask
    & breakfast_candidate_mask
)

technically_valid_main_meal_mask = (
    technically_valid_mask
    & refined_main_meal_mask
)

valid_product_counts = pd.Series(
    {
        "Технически пригодные breakfast candidates": (
            technically_valid_breakfast_mask.sum()
        ),
        "Технически пригодные main_meal candidates": (
            technically_valid_main_meal_mask.sum()
        ),
    },
    name="Количество рецептов",
).astype(int)

display(valid_product_counts.to_frame())

,Количество рецептов
Технически пригодные breakfast candidates,36304
Технически пригодные main_meal candidates,87545


## 7. Проверка `RecipeIngredientParts` ↔ `RecipeIngredientQuantities`

Food.com хранит ингредиенты и количества в отдельных массивах. Для grocery-list
и масштабирования рецептов было бы удобно считать их позиционно связанными,
поэтому это предположение проверяется отдельно.

In [14]:
ingredient_lists = (
    recipes["RecipeIngredientParts"]
    .apply(to_list)
)
quantity_lists = (
    recipes["RecipeIngredientQuantities"]
    .apply(to_list)
)

ingredient_lengths = ingredient_lists.str.len()
quantity_lengths = quantity_lists.str.len()
length_mismatch_mask = (
    ingredient_lengths
    .ne(quantity_lengths)
)

print(
    "Рецептов с разной длиной массивов: "
    f"{int(length_mismatch_mask.sum()):,}"
)

length_statistics = (
    ingredient_lengths
    .quantile([0, 0.5, 0.95, 1])
    .rename(
        index={
            0.0: "Минимум",
            0.5: "Медиана",
            0.95: "95%",
            1.0: "Максимум",
        }
    )
    .astype(int)
)

display(
    length_statistics
    .rename("Длина списка ингредиентов")
    .to_frame()
)

comparison = pd.DataFrame(
    {
        "Name": recipes["Name"],
        "RecipeIngredientParts": ingredient_lists,
        "RecipeIngredientQuantities": quantity_lists,
        "IngredientCount": ingredient_lengths,
        "QuantityCount": quantity_lengths,
    }
)

mismatch_count = int(length_mismatch_mask.sum())
if mismatch_count:
    display(
        comparison.loc[length_mismatch_mask]
        .sample(
            n=min(5, mismatch_count),
            random_state=42,
        )
    )

Рецептов с разной длиной массивов: 407,536


,Длина списка ингредиентов
Минимум,0
Медиана,7
95%,15
Максимум,44


,Name,RecipeIngredientParts,RecipeIngredientQuantities,IngredientCount,QuantityCount
419279,Blueberry Breakfast Braid,"[all-purpose flour, sugar, salt, ground cinnam...","[5 - 5 1⁄2, 1⁄2, 2, 1, 1⁄2, 1, 1⁄2, 1⁄2, 2, 1...",12,15
273927,Shrimp Etouffee,"[butter, onions, celery, bell peppers, garlic ...","[1 1⁄4, 4, 5, 5, 5, 1 1⁄4, 1 1⁄4, 2, 2, 5, 5, ...",12,13
360918,Surprise Burgers,"[fresh chives, ground beef, olive oil, gruyere...","[1⁄2, 3, 1 1⁄2, 2, None, 4, 4, 1⁄2, 4, 1, 1]",7,11
477146,Parmesan Chicken With Caesar-Roasted Romaine,"[chicken breast halves, parmesan cheese, panko...","[4, None, 1⁄2, 1⁄2, 3, 2, 4, 2, 4 -8, 2, 1]",10,11
246979,Chicken Wellingtons,"[chicken thigh fillets, puff pastry, egg, drie...","[4 -6, 2 -3, 2 -3, 1, None, None]",4,6


### Решение для v1

Позиционное соответствие `RecipeIngredientParts[i]` и
`RecipeIngredientQuantities[i]` ненадёжно для большой части датасета.

Поэтому:
- КБЖУ берётся из готовых per-serving nutrition values Food.com;
- `RecipeIngredientParts` используется для предпочтений и исключений;
- `RecipeIngredientQuantities` не участвует в core planning pipeline v1;
- нормализация quantities / grocery list переносится в отдельную будущую задачу.

## 8. Словарь ингредиентов

In [15]:
ingredient_list_empty_mask = (
    ingredient_lists
    .str.len()
    .eq(0)
)

ingredient_values_raw = ingredient_lists.explode()
ingredient_values = (
    ingredient_values_raw
    .dropna()
    .astype(str)
    .str.strip()
)
empty_ingredient_value_count = (
    ingredient_values.eq("").sum()
)
ingredient_values = ingredient_values[
    ingredient_values.ne("")
]

normalized_ingredient_values = (
    ingredient_values
    .str.casefold()
)

ingredient_summary = pd.Series(
    {
        "Уникальных ингредиентов (исходное написание)": (
            ingredient_values.nunique()
        ),
        "Уникальных после нормализации регистра/пробелов": (
            normalized_ingredient_values.nunique()
        ),
        "Рецептов с null/empty списком ингредиентов": (
            ingredient_list_empty_mask.sum()
        ),
        "Пустых строк внутри массивов": (
            empty_ingredient_value_count
        ),
    },
    name="Количество",
).astype(int)

display(ingredient_summary.to_frame())

ingredient_frequencies = (
    ingredient_values
    .value_counts()
)

print("Топ-30 самых частых ингредиентов:")
display(
    ingredient_frequencies
    .head(30)
    .rename_axis("Ингредиент")
    .rename("Количество рецептов")
    .to_frame()
)

print("20 случайных уникальных названий:")
display(
    ingredient_values
    .drop_duplicates()
    .sample(
        n=min(
            20,
            ingredient_values.nunique(),
        ),
        random_state=42,
    )
    .reset_index(drop=True)
    .rename("Ингредиент")
    .to_frame()
)

,Количество
Уникальных ингредиентов (исходное написание),7368
Уникальных после нормализации регистра/пробелов,7309
Рецептов с null/empty списком ингредиентов,2219
Пустых строк внутри массивов,0


Топ-30 самых частых ингредиентов:


,Количество рецептов
Ингредиент,
salt,199781
butter,136080
sugar,115852
onion,87546
water,84376
eggs,81187
olive oil,76724
flour,62410
milk,61247


20 случайных уникальных названий:


,Ингредиент
0,boneless beef top round steak
1,mocha ice cream
2,bosc pears
3,durian
4,Pesto
5,dried chinese dates
6,Tomato Confit
7,raw beet
8,andouille turkey sausage
9,star fruit


In [16]:
frequency_groups = pd.Series(
    {
        "Встречаются ровно 1 раз": ingredient_frequencies.eq(1).sum(),
        "Встречаются ≤ 5 раз": ingredient_frequencies.le(5).sum(),
        "Встречаются ≤ 10 раз": ingredient_frequencies.le(10).sum(),
        "Встречаются ≤ 100 раз": ingredient_frequencies.le(100).sum(),
        "Встречаются > 100 раз": ingredient_frequencies.gt(100).sum(),
        "Встречаются > 1000 раз": ingredient_frequencies.gt(1000).sum(),
    },
    name="Количество уникальных ингредиентов",
).astype(int)

frequency_statistics = (
    ingredient_frequencies
    .quantile(
        [0.50, 0.90, 0.95, 0.99, 1.00]
    )
    .rename(
        index={
            0.50: "Медиана",
            0.90: "90-й перцентиль",
            0.95: "95-й перцентиль",
            0.99: "99-й перцентиль",
            1.00: "Максимум",
        }
    )
    .rename("Частота ингредиента")
)

display(frequency_groups.to_frame())
display(frequency_statistics.to_frame())

frequency_thresholds = [
    1,
    10,
    50,
    100,
    500,
    1000,
]
total_ingredient_mentions = int(
    ingredient_frequencies.sum()
)

coverage_rows = []
for threshold in frequency_thresholds:
    frequent = ingredient_frequencies[
        ingredient_frequencies.ge(threshold)
    ]
    covered_mentions = int(frequent.sum())

    coverage_rows.append(
        {
            "Порог частоты": f"frequency >= {threshold}",
            "Количество уникальных ингредиентов": len(frequent),
            "Количество всех их упоминаний": covered_mentions,
            "% от всех ingredient mentions": (
                covered_mentions
                / total_ingredient_mentions
                * 100
            ),
        }
    )

ingredient_coverage = (
    pd.DataFrame(coverage_rows)
    .set_index("Порог частоты")
)
ingredient_coverage[
    "% от всех ingredient mentions"
] = (
    ingredient_coverage[
        "% от всех ingredient mentions"
    ].round(2)
)

print(
    "Всего упоминаний ингредиентов: "
    f"{total_ingredient_mentions:,}"
)
display(ingredient_coverage)

,Количество уникальных ингредиентов
Встречаются ровно 1 раз,1632
Встречаются ≤ 5 раз,2793
Встречаются ≤ 10 раз,3390
Встречаются ≤ 100 раз,5429
Встречаются > 100 раз,1939
Встречаются > 1000 раз,522


,Частота ингредиента
Медиана,15.00
90-й перцентиль,609.00
95-й перцентиль,1590.65
99-й перцентиль,9904.19
Максимум,199781.00


Всего упоминаний ингредиентов: 4,129,433


,Количество уникальных ингредиентов,Количество всех их упоминаний,% от всех ingredient mentions
Порог частоты,,,
frequency >= 1,7368,4129433,100.00
frequency >= 10,4080,4120591,99.79
frequency >= 50,2548,4083263,98.88
frequency >= 100,1944,4040634,97.85
frequency >= 500,829,3783875,91.63
frequency >= 1000,524,3567802,86.40


## 9. Почему substring matching недостаточен

Простое совпадение по подстроке смешивает разные сущности:
`egg` → `eggplant`, `butter` → `buttermilk` / `peanut butter`,
`rice` → `rice vinegar`, `chicken` → `chicken broth`.

Это обосновывает отдельный semantic-normalization layer для ингредиентов.

In [17]:
base_words = [
    "egg",
    "garlic",
    "onion",
    "chicken",
    "beef",
    "milk",
    "cheese",
    "butter",
    "rice",
    "tomato",
]

for base_word in base_words:
    contains_word = (
        ingredient_frequencies
        .index
        .astype(str)
        .str.contains(
            base_word,
            case=False,
            regex=False,
        )
    )

    variants = (
        ingredient_frequencies[contains_word]
        .head(10)
        .rename("Частота")
        .rename_axis("Вариант ингредиента")
        .to_frame()
    )

    print(
        f"{base_word}: "
        f"{int(contains_word.sum())} уникальных вариантов "
        "(показаны top-10)"
    )
    display(variants)

egg: 55 уникальных вариантов (показаны top-10)


,Частота
Вариант ингредиента,
eggs,81187
egg,39773
hard-boiled eggs,2444
eggplant,1908
parmigiano-reggiano cheese,1739
egg substitute,1664
eggplants,1316
Egg Beaters egg substitute,595
hard-boiled egg,418


garlic: 114 уникальных вариантов (показаны top-10)


,Частота
Вариант ингредиента,
garlic cloves,59604
garlic,35686
garlic powder,23341
garlic clove,23111
fresh garlic,1774
minced garlic cloves,1409
fresh garlic cloves,819
minced garlic clove,720
garlic paste,524


onion: 101 уникальных вариантов (показаны top-10)


,Частота
Вариант ингредиента,
onion,87546
onions,22271
red onion,13442
green onions,12778
green onion,12707
onion powder,7855
yellow onion,5697
sweet onion,2919
red onions,2570


... (10 outputs left out)

rice: 114 уникальных вариантов (показаны top-10)


,Частота
Вариант ингредиента,
rice,3882
cooked rice,3492
rice vinegar,3474
brown rice,1667
white rice,1456
long grain rice,1353
basmati rice,1225
arborio rice,1022
cooked brown rice,910


tomato: 226 уникальных вариантов (показаны top-10)


,Частота
Вариант ингредиента,
tomatoes,27086
tomato sauce,10417
tomato paste,10141
diced tomatoes,8407
cherry tomatoes,4210
crushed tomatoes,2742
plum tomatoes,2670
stewed tomatoes,2034
roma tomatoes,2020


## 10. Решения для Meal Generator v1

По результатам EDA фиксируется следующий контракт данных.

### Recipe domain model

**Identity**
- `recipe_id`
- `name`
- `source`

**Classification**
- `meal_types`
- `source_category`
- `keywords`

**Nutrition per serving**
- `calories`
- `protein`
- `fat`
- `carbs`

**Preparation**
- `servings`
- `total_time`
- `ingredients`
- `instructions`

**Quality**
- `rating`
- `review_count`

### Принятые решения

1. Food.com используется как основной источник рецептов.
2. В core nutrition v1 используется стандартное КБЖУ на порцию.
3. Из временных признаков приложению нужен только `TotalTime`.
4. Rating и ReviewCount используются как soft ranking signals; отсутствие рейтинга
   не делает рецепт непригодным.
5. `Breakfast` / `Brunch` дают high-confidence breakfast candidates.
6. Для `main_meal` используется воспроизводимый weak-labeling baseline; полноценная
   классификация meal types может быть улучшена позже.
7. Для v1 требуются валидные ingredients, instructions, servings и КБЖУ.
8. `RecipeIngredientQuantities` исключён из core pipeline из-за ненадёжного
   соответствия массиву ингредиентов.
9. Названия ингредиентов требуют semantic normalization перед использованием
   в пользовательских exclusions/preferences.
10. Следующий этап — вынести эти правила из EDA в воспроизводимый
    preprocessing pipeline и сформировать curated recipe corpus.